In [5]:
import pandas as pd
import pickle
import joblib
with open("/content/drive/MyDrive/Projects/IGEM Summer 2026/Therapeutic Antibody Humanization Prediction /model.pkl", "rb") as f:
    model = joblib.load(f)
with open("/content/drive/MyDrive/Projects/IGEM Summer 2026/Therapeutic Antibody Humanization Prediction /feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)

hotspots = {
    12: "T",
    14: "T",
    19: "L",
    20: "S",
    21: "C"
}

In [6]:
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

def extract_features(seq):

    return {
        aa: seq.count(aa)/len(seq)
        for aa in AMINO_ACIDS
    }

In [7]:
def humanization_score(seq):

    X = pd.DataFrame(
        [extract_features(seq)]
    )

    return model.predict_proba(X)[0][1]

In [10]:
df = pd.read_csv("/content/drive/MyDrive/Projects/IGEM Summer 2026/Therapeutic Antibody Humanization Prediction /humanization_scores.csv")

print(df.shape)
df.head()

seq = df.iloc[0]["sequence"]

print(
    humanization_score(seq)
)

(23022, 5)
0.745


In [14]:
!pip install abnumber

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 125.9 MB/s eta 0:00:00


In [15]:
from abnumber import Chain

def get_fr1(seq):

    chain = Chain(
        seq,
        scheme="imgt"
    )

    return str(chain.fr1_seq)

In [16]:
def suggest_mutations(seq):

    fr1 = get_fr1(seq)

    suggestions = []

    for pos, human_residue in hotspots.items():

        if pos >= len(fr1):
            continue

        current = fr1[pos]

        if current != human_residue:

            suggestions.append(
                {
                    "position": pos,
                    "current": current,
                    "suggested": human_residue
                }
            )

    return suggestions

In [17]:
def mutate_fr1(fr1, pos, aa):

    fr1 = list(fr1)

    fr1[pos] = aa

    return "".join(fr1)

In [18]:
def score_mutations(seq):

    original_score = humanization_score(seq)

    suggestions = suggest_mutations(seq)

    return {
        "score": original_score,
        "suggestions": suggestions
    }

In [20]:
!apt-get install -y hmmer

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libdivsufsort3
Suggested packages:
  hmmer-doc
The following NEW packages will be installed:
  hmmer libdivsufsort3
0 upgraded, 2 newly installed, 0 to remove and 3 not upgraded.
Need to get 1,198 kB of archives.
After this operation, 7,621 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libdivsufsort3 amd64 2.0.1-5 [42.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 hmmer amd64 3.3.2+dfsg-1 [1,155 kB]
Fetched 1,198 kB in 0s (9,628 kB/s)
Selecting previously unselected package libdivsufsort3:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libdivsufsort3_2.0.1-5_amd64.deb ...
Unpacking libdivsufsort3:amd64 (2.0.1-5) ...
Selecting previously unselected package hmmer.
Preparing to unpack .../hmmer_3.3.2+dfsg-1_amd

{'score': np.float64(1.0), 'suggestions': []}

In [21]:
test_seq = df.iloc[100]["sequence"]

result = score_mutations(
    test_seq
)

result

{'score': np.float64(1.0), 'suggestions': []}